# 🏭 Smart Factory — Pipeline Analisi Incidenti v2
**Università La Sapienza — Assignment 1**

Analisi audio multi-fonte con Gemini 2.5 Flash → Ontologia → Excel

---
## ⚙️ Due modalità di estrazione a confronto

| Modalità | Descrizione | Output Excel |
|---|---|---|
| **MODALITÀ A** — Singolo audio | Ogni audio viene analizzato separatamente da Gemini → 1 tabella per audio | `report_singolo_NOMEAUDIO.xlsx` per ciascun file |
| **MODALITÀ B** — Fusione | Fonde in Python i risultati già estratti dalla Modalità A → 1 tabella unificata, **senza nuove chiamate API** | `report_multiaudio.xlsx` |

> **Ordine di esecuzione:** esegui sempre prima la Cella 13 (Modalità A), poi la Cella 14 (Modalità B), poi la Cella 15 (Confronto).

In [1]:
# =====================================================================
# CELLA 1: INSTALLAZIONE — esegui UNA SOLA VOLTA, poi riavvia il kernel
# =====================================================================
%pip install google-generativeai pandas openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# =====================================================================
# CELLA 2: IMPORT
# =====================================================================
import os
import re
import time
import json
import pandas as pd
import google.generativeai as genai
#formattazioen visiva di excel
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.utils import get_column_letter
from IPython.display import display, Markdown

print('✅ Import completati correttamente.')

c:\Users\flavc\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\flavc\AppData\Local\Temp\ipykernel_20220\2525539176.py:9: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


✅ Import completati correttamente.


In [ ]:
# =====================================================================
# CELLA 3: CONFIGURAZIONE
# SICUREZZA: imposta la chiave come variabile d'ambiente:
#   Windows:   set GEMINI_API_KEY=la_tua_chiave
#   Mac/Linux: export GEMINI_API_KEY=la_tua_chiave
# In alternativa scrivi la chiave qui solo per test locali.
# =====================================================================

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "INSERISCI_QUI_LA_CHIAVE")
genai.configure(api_key=GEMINI_API_KEY)

# ---------------------------------------------------------------
# 🎯 LISTA FILE AUDIO — aggiungi o rimuovi percorsi liberamente!
#    Puoi usare percorsi assoluti (r"C:\...") o relativi ("audio.mp3")
#    Formati supportati: .mp3, .wav, .m4a, .ogg, .flac
# ---------------------------------------------------------------
FILE_AUDIO = [
    "Elena RSPP.mp3",
    "Marco Capolinea.mp3",
    "Martina Operatrice.mp3",
    # Aggiungi altri file qui sotto se necessario:
    # r"percorso\audio4.mp3",
    # r"percorso\audio5.wav",
]

# Cartella di output (verrà creata se non esiste)
OUTPUT_DIR = "output_report"
os.makedirs(OUTPUT_DIR, exist_ok=True)

#il ciclo stampa per ogni file uno stato di verifica se esiste o meno, così da avere un riscontro immediato prima di procedere con l'elaborazione
print(f'✅ Configurazione completata. {len(FILE_AUDIO)} file audio configurati.')
for i, f in enumerate(FILE_AUDIO, 1):
    esiste = "✅" if os.path.exists(f) else "❌ NON TROVATO"
    print(f"   Audio {i}: {os.path.basename(f)} — {esiste}")

✅ Configurazione completata. 3 file audio configurati.
   Audio 1: Elena RSPP.mp3 — ✅
   Audio 2: Marco Capolinea.mp3 — ✅
   Audio 3: Martina Operatrice.mp3 — ✅


In [ ]:
# =====================================================================
# CELLA 4: PROMPT DI SISTEMA
# Due prompt distinti per le due modalità.
# =====================================================================

# --- STRUTTURA JSON CONDIVISA (ontologia) ---
_SCHEMA_JSON = """
{
  "Evento": {
    "id_evento": "INC-AAAA-NNN",
    "Data_Ora": "GG/MM/AAAA HH:MM",
    "Tipo_Evento": "Incidente | Near Miss | Guasto Tecnico | Manutenzione",
    "Gravita_Stimata": "Bassa | Media | Alta | Critica | Fatale",
    "Prima_dell_evento": "Descrizione stato macchina e lavoratore",
    "Trigger_Evento_Principale": "Causa scatenante immediata",
    "Dinamica_Sintesi": "Riassunto neutro e oggettivo",
  },
  "Luoghi_e_Mezzi": {
    "Reparto": "Nome univoco del reparto (es. Reparto Stampa, Linea 3). Scrivi N/D se non menzionato.",
    "Area_Coinvolta": "Zona specifica all'interno del reparto (es. Zona pressa, Uscita nastro). Scrivi N/D se non menzionata.",
    "Macchinario_Coinvolto": "Nome tecnico del macchinario (es. Pressa idraulica P200, Tornio CNC). Scrivi N/D se non menzionato.",
    "Codice_Macchinario": "Codice o targa del macchinario se citato, altrimenti N/D.",
    "Condizioni_Ambientali": "Condizioni fisiche dell'ambiente al momento dell'evento (es. pavimento bagnato, scarsa illuminazione). Scrivi N/D se non menzionate."
  },
  "Analisi_Cause": {
    "Causa_Tecnica": "",
    "Fattore_Umano": "",
    "Fattore_Ambientale": "",
    "Causa_Radice": "Root cause"
  },
  "Conseguenze": {
    "Conseguenza_Fisica_Immediata": "",
    "Parte_Corpo_Lesa": "",
    "Giorni_Inabilita_Stimati": "",
    "Conseguenza_Materiale_Aziendale": "",
    "Fermo_Produzione_Ore": ""
  },
  "Azioni_Correttive": {
    "Procedura_Corretta_Violata": "",
    "DPI_Mancanti_o_Inadeguati": "",
    "Azione_Correttiva_Immediata": "",
    "Azione_Correttiva_da_Implementare": "",
    "Responsabile_Azione": "",
    "Scadenza_Azione": ""
  },
  "Persone_Coinvolte": [
    {
      "Nome": "",
      "Ruolo_Aziendale": "",
      "Coinvolgimento": "Vittima | Testimone | RSPP | Supervisore",
      "Danno_Riportato": "",
      "Affidabilita_Testimonianza": "Alta | Media | Bassa",
      "Note_Testimonianza": ""
    }
  ]
}
"""

# ---- PROMPT MODALITÀ A: singolo audio ----
def build_prompt_singolo(nome_file, indice_totale):
    """Genera il prompt per l'analisi di un singolo audio."""
    return f"""
Ruolo: Sei un esperto HSE (Health, Safety & Environment) e Data Extraction.
Ricevi UNA registrazione audio (file {indice_totale}: '{nome_file}') relativa a un incidente industriale.

Istruzioni:
1. Estrai tutte le informazioni contenute in questo singolo audio.
2. Per i campi non menzionati nel racconto, scrivi esattamente "N/D".
3. Non inventare o inferire informazioni non presenti nell'audio. #riduce le allucinazioni
4. Usa terminologia tecnica HSE corretta; se il parlante usa gergo, traduci.

Istruzioni specifiche per Luoghi_e_Mezzi:
- Reparto: estrai il nome esatto del reparto citato dal parlante. Se usa un soprannome o abbreviazione, riportala così com'è senza modifiche.
- Area_Coinvolta: estrai la zona precisa all'interno del reparto (es. vicino alla pressa, zona scarico). Se non citata esplicitamente, scrivi N/D.
- Macchinario_Coinvolto: estrai il nome tecnico o il nome comune usato dal parlante per il macchinario. Se usa un soprannome (es. 'la bestia', 'il tornio vecchio'), riportalo e affianca il termine tecnico se deducibile dal contesto.
- Codice_Macchinario: solo se citato esplicitamente (targa, codice alfanumerico). Altrimenti N/D.
Macchinario_Coinvolto e Area_Coinvolta non possono essere uguali

Formato output:
Restituisci ESCLUSIVAMENTE un oggetto JSON valido (nessun testo prima/dopo, nessun backtick).
Schema:
{_SCHEMA_JSON}
"""

# ---- PROMPT MODALITÀ B: N audio insieme ----
def build_prompt_multi(risultati_singoli):
    """Genera il prompt per la fusione intelligente dei JSON estratti."""
    
    # Convertiamo i risultati della Modalità A in una stringa leggibile per Gemini
    testi_report = ""
    for i, r in enumerate(risultati_singoli):
        testi_report += f"\n--- REPORT AUDIO {i+1} ({r['nome']}) ---\n"
        testi_report += json.dumps(r["dati"], indent=2, ensure_ascii=False)

    return f"""
Ruolo: Sei un Senior HSE Specialist esperto in investigazione di incidenti.
Hai analizzato singolarmente 3 testimonianze. Ora devi redigere il RAPPORTO FINALE UNIFICATO.

ECCO I REPORT PARZIALI DA UNIRE:
{testi_report}

ISTRUZIONI RIGOROSE PER LA SINTESI:
1. RISPOSTA UNICA: Per ogni campo (es. Macchinario, Trigger), fornisci una sola descrizione sintetica. NON usare " | " o elenchi di versioni diverse.
2. PRIORITÀ TECNICA: Se le fonti usano nomi diversi, usa la terminologia tecnica del RSPP o quella normativa HSE più corretta.
3. RICONCILIAZIONE: Se una fonte è incompleta e l'altra è dettagliata, usa la versione dettagliata per arricchire il dato unico.
4. DEDUPLICAZIONE PERSONE: Crea una lista unica di 'Persone_Coinvolte'. Fondi le informazioni (ruolo, danno) in un unico oggetto per persona.
5. Se un dato è totalmente assente in tutti i report, scrivi "N/D".

Formato output:
Restituisci ESCLUSIVAMENTE un oggetto JSON valido secondo lo schema:
{_SCHEMA_JSON}
"""

print("✅ Prompt Modalità B (AI Fusion) configurato.")



✅ Prompt Modalità B (AI Fusion) configurato.


In [5]:
# =====================================================================
# CELLA 5: UTILITY GEMINI — upload, attesa, cleanup
# =====================================================================

def _upload_e_attendi(percorsi):
    """Carica una lista di file su Gemini Files API e attende che siano pronti.
    Restituisce la lista di oggetti file Gemini."""
    file_caricati = []
    for percorso in percorsi:
        if not os.path.exists(percorso):
            raise FileNotFoundError(f"❌ File non trovato: '{percorso}'")
        print(f"   ⬆️  Upload: {os.path.basename(percorso)}")
        file_caricati.append(genai.upload_file(path=percorso))

    print("   ⏳ Attendo elaborazione", end="")
    for i, f in enumerate(file_caricati):
        while f.state.name == "PROCESSING":
            print(".", end="", flush=True)
            time.sleep(2)
            file_caricati[i] = genai.get_file(f.name)
            f = file_caricati[i]
    print(" ✅")
    return file_caricati


def _chiedi_gemini(prompt, file_gemini):
    """Invia prompt + file a Gemini e restituisce il JSON parsato."""
    model = genai.GenerativeModel(
        model_name="gemini-2.5-flash",
        generation_config={"response_mime_type": "application/json"}
    )
    risposta = model.generate_content([prompt] + file_gemini)
    testo = re.sub(r"```json\s*|\s*```", "", risposta.text).strip()
    return json.loads(testo)


def _cleanup(file_gemini):
    """Elimina i file dal cloud Gemini dopo l'uso."""
    for f in file_gemini:
        try:
            genai.delete_file(f.name)
        except Exception:
            pass

print("✅ Utility Gemini pronte.")

✅ Utility Gemini pronte.


In [6]:
# =====================================================================
# CELLA 6: MODALITÀ A — analisi singola per ciascun audio
#ogni file viene processato uno alla volta in sequenza, con un report dedicato per ciascuno. Questa modalità è più lenta ma riduce le interferenze tra fonti diverse.
# =====================================================================

def analizza_singoli(lista_percorsi):
    """Analizza ogni audio INDIVIDUALMENTE.
    Ritorna una lista di dict: [{"percorso": ..., "dati": {...}}, ...]"""
    risultati = []
    n = len(lista_percorsi)
    for idx, percorso in enumerate(lista_percorsi, 1):
        nome = os.path.basename(percorso)
        print(f"\n🎙️ [{idx}/{n}] Analisi singola: '{nome}'")
        file_caricati = _upload_e_attendi([percorso])
        print(f"   🧠 Estrazione dati...")
        prompt = build_prompt_singolo(nome, idx)
        dati = _chiedi_gemini(prompt, file_caricati)
        _cleanup(file_caricati)
        dati["_meta"] = {"fonte": nome, "indice": idx, "modalita": "singolo"}
        risultati.append({"percorso": percorso, "nome": nome, "dati": dati})
        print(f"   ✅ Completato.")
    print(f"\n✅ Modalità A completata: {n} audio analizzati singolarmente.")
    return risultati

print("✅ Funzione modalità A pronta.")

✅ Funzione modalità A pronta.


In [7]:
# =====================================================================
# CELLA 7: MODALITÀ B — fusione dei dati già estratti dalla Modalità A
# Non effettua nuove chiamate a Gemini: lavora sui risultati_singoli
# già in memoria, fondendoli in un unico dict con logica Python.
# =====================================================================

def fondi_risultati_singoli(risultati_singoli):
    """Chiama Gemini per fondere i JSON ottenuti dalla Modalità A."""
    if not risultati_singoli:
        raise ValueError("Lista risultati vuota. Esegui prima la Modalità A.")

    print(f"\n🧠 Avvio fusione intelligente di {len(risultati_singoli)} fonti con Gemini...")

    # 1. Prepariamo il prompt usando la funzione della Cella 4
    prompt_fusione = build_prompt_multi(risultati_singoli)
    
    # 2. Chiamata a Gemini 1.5 Flash
    model = genai.GenerativeModel("gemini-2.5-flash")
    risposta = model.generate_content(prompt_fusione)
    
    try:
        # Pulizia backticks markdown e parsing JSON
        testo_pulito = risposta.text.strip().replace("```json", "").replace("```", "")
        return json.loads(testo_pulito)
    except Exception as e:
        print(f"❌ Errore nel parsing della fusione: {e}")
        return risultati_singoli[0]["dati"] # Fallback sul primo report in caso di errore

print("✅ Funzione fondi_risultati_singoli pronta.")

✅ Funzione fondi_risultati_singoli pronta.


In [8]:
# =====================================================================
# CELLA 8: UTILITY EXCEL — stili e funzione di formattazione
# =====================================================================

# Stili globali riutilizzabili
HEADER_FONT  = Font(bold=True, size=11, color="FFFFFF", name="Arial")
HEADER_FILL  = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
CELL_FONT    = Font(size=10, name="Arial")
ALT_FILL     = PatternFill(start_color="EBF5FB", end_color="EBF5FB", fill_type="solid")
WRAP_TOP     = Alignment(wrap_text=True, vertical="top")
CENTER       = Alignment(horizontal="center", vertical="center", wrap_text=True)
THIN         = Side(style="thin", color="BFBFBF")
BORDER       = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)


def _applica_stile(ws, larghezze=None):
    """Applica intestazioni, righe alternate, bordi e larghezze a un foglio."""
    for cell in ws[1]:
        cell.font, cell.fill, cell.alignment, cell.border = HEADER_FONT, HEADER_FILL, CENTER, BORDER
    for row_idx, row in enumerate(ws.iter_rows(min_row=2), start=2):
        for cell in row:
            cell.font, cell.alignment, cell.border = CELL_FONT, WRAP_TOP, BORDER
            if row_idx % 2 == 0:
                cell.fill = ALT_FILL
    for i, col in enumerate(ws.columns):
        letter = get_column_letter(col[0].column)
        if larghezze and i < len(larghezze):
            ws.column_dimensions[letter].width = larghezze[i]
        else:
            max_len = max((len(str(c.value or "")) for c in col), default=10)
            ws.column_dimensions[letter].width = min(max(max_len * 0.85, 15), 55)
    ws.freeze_panes = "A2"
    ws.row_dimensions[1].height = 32


def _dati_a_dataframes(dati):
    """Converte il dict JSON estratto in dizionario di DataFrame (uno per sezione)."""
    ev = dati.get("Evento", {})
    lm = dati.get("Luoghi_e_Mezzi", {})
    ac = dati.get("Analisi_Cause", {})
    co = dati.get("Conseguenze", {})
    az = dati.get("Azioni_Correttive", {})
    pc = dati.get("Persone_Coinvolte", [])

    frames = {
        "EVENTO": pd.DataFrame([{
            "ID Evento":             ev.get("id_evento", "N/D"),
            "Data / Ora":            ev.get("Data_Ora", "N/D"),
            "Tipo Evento":           ev.get("Tipo_Evento", "N/D"),
            "Gravità":               ev.get("Gravita_Stimata", "N/D"),
            "Situazione Pre-Evento": ev.get("Prima_dell_evento", "N/D"),
            "Trigger Principale":    ev.get("Trigger_Evento_Principale", "N/D"),
            "Dinamica (Sintesi)":    ev.get("Dinamica_Sintesi", "N/D"),
            }]),
        "LUOGHI & MEZZI": pd.DataFrame([{
            "Reparto":               lm.get("Reparto", "N/D"),
            "Area Coinvolta":        lm.get("Area_Coinvolta", "N/D"),
            "Macchinario":           lm.get("Macchinario_Coinvolto", "N/D"),
            "Codice Macchinario":    lm.get("Codice_Macchinario", "N/D"),
            "Condizioni Ambientali": lm.get("Condizioni_Ambientali", "N/D"),
        }]),
        "ANALISI CAUSE": pd.DataFrame([{
            "Causa Tecnica":              ac.get("Causa_Tecnica", "N/D"),
            "Fattore Umano":              ac.get("Fattore_Umano", "N/D"),
            "Fattore Ambientale":         ac.get("Fattore_Ambientale", "N/D"),
            "Causa Radice (Root Cause)":  ac.get("Causa_Radice", "N/D"),
        }]),
        "CONSEGUENZE": pd.DataFrame([{
            "Conseguenza Fisica":          co.get("Conseguenza_Fisica_Immediata", "N/D"),
            "Parte del Corpo Lesa":        co.get("Parte_Corpo_Lesa", "N/D"),
            "Giorni Inabilità (Stimati)":  co.get("Giorni_Inabilita_Stimati", "N/D"),
            "Conseguenza Materiale":       co.get("Conseguenza_Materiale_Aziendale", "N/D"),
            "Fermo Produzione (ore)":      co.get("Fermo_Produzione_Ore", "N/D"),
        }]),
        "AZIONI CORRETTIVE": pd.DataFrame([{
            "Procedura Violata":                  az.get("Procedura_Corretta_Violata", "N/D"),
            "DPI Mancanti/Inadeguati":            az.get("DPI_Mancanti_o_Inadeguati", "N/D"),
            "Azione Correttiva Immediata":        az.get("Azione_Correttiva_Immediata", "N/D"),
            "Azione Correttiva da Implementare":  az.get("Azione_Correttiva_da_Implementare", "N/D"),
            "Responsabile":                       az.get("Responsabile_Azione", "N/D"),
            "Scadenza":                           az.get("Scadenza_Azione", "N/D"),
        }]),
        "PERSONE COINVOLTE": (
            pd.DataFrame(pc).rename(columns=lambda c: c.replace("_", " "))
            if pc else
            pd.DataFrame(columns=["Nome", "Ruolo Aziendale", "Coinvolgimento",
                                   "Danno Riportato", "Affidabilita Testimonianza", "Note Testimonianza"])
        ),
    }
    return frames


# Larghezze colonne per foglio (in caratteri)
_LARGHEZZE = {
    "EVENTO":            [15, 18, 18, 12, 50, 50, 60, 50],
    "LUOGHI & MEZZI":    [25, 25, 30, 20, 35],
    "ANALISI CAUSE":     [45, 45, 35, 55],
    "CONSEGUENZE":       [40, 25, 22, 45, 22],
    "AZIONI CORRETTIVE": [45, 35, 45, 55, 25, 18],
    "PERSONE COINVOLTE": [20, 22, 18, 35, 22, 45],
}

print("✅ Utility Excel pronte.")


✅ Utility Excel pronte.


In [9]:
# =====================================================================
# CELLA 9: SALVATAGGIO EXCEL — Modalità A (un file per audio)
#per ogni risultato della Modalità A crea un file .xlsx separato
# =====================================================================

def salva_excel_singolo(risultato):
    """Salva un Excel per un singolo risultato (da analisi_singoli).
    risultato = {"percorso": ..., "nome": ..., "dati": {...}}
    """
    nome_base = os.path.splitext(risultato["nome"])[0]
    # Pulizia nome file da caratteri non validi
    nome_safe = re.sub(r'[<>:"/\\|?*]', '_', nome_base)
    file_output = os.path.join(OUTPUT_DIR, f"report_singolo_{nome_safe}.xlsx")

    frames = _dati_a_dataframes(risultato["dati"])

    with pd.ExcelWriter(file_output, engine="openpyxl") as writer:
        for sheet_name, df in frames.items():
            df.to_excel(writer, sheet_name=sheet_name, index=False)
        wb = writer.book
        for name in wb.sheetnames:
            _applica_stile(wb[name], _LARGHEZZE.get(name))

    print(f"   💾 Salvato: '{file_output}'")
    return file_output


def salva_excel_tutti_singoli(risultati):
    """Salva un file Excel per ciascun risultato della Modalità A."""
    print(f"\n📊 Modalità A — salvataggio {len(risultati)} file Excel individuali...")
    file_salvati = []
    for r in risultati:
        file_salvati.append(salva_excel_singolo(r))
    print(f"✅ {len(file_salvati)} file Excel salvati nella cartella '{OUTPUT_DIR}'.")
    return file_salvati

print("✅ Funzione salvataggio Modalità A pronta.")

✅ Funzione salvataggio Modalità A pronta.


In [10]:
# =====================================================================
# CELLA 10: SALVATAGGIO EXCEL — Modalità B (file unico multi-fonte)
#salva il report unificato della Modalità B in un singolo file .xlsx
# =====================================================================

def salva_excel_multi(dati, file_output=None):
    """Salva un Excel unificato dal risultato della Modalità B."""
    if file_output is None:
        file_output = os.path.join(OUTPUT_DIR, "report_multiaudio.xlsx")

    print(f"\n📊 Modalità B — salvataggio report multi-audio: '{file_output}'...")
    frames = _dati_a_dataframes(dati)

    with pd.ExcelWriter(file_output, engine="openpyxl") as writer:
        for sheet_name, df in frames.items():
            df.to_excel(writer, sheet_name=sheet_name, index=False)
        wb = writer.book
        for name in wb.sheetnames:
            _applica_stile(wb[name], _LARGHEZZE.get(name))

    print(f"✅ Report multi-audio salvato: '{file_output}'")
    return file_output

print("✅ Funzione salvataggio Modalità B pronta.")

✅ Funzione salvataggio Modalità B pronta.


In [11]:
# =====================================================================
# CELLA 11: ANTEPRIMA IN JUPYTER
# =====================================================================

# stampa un titolo Markdown e il DataFrame dell'Evento per ciascun audio, permettendo un controllo rapido dell'estrazione singola
def mostra_anteprima_singoli(risultati):
    """Mostra una tabella EVENTO per ciascun audio analizzato singolarmente."""
    display(Markdown("## 📋 MODALITÀ A — Anteprima per audio"))
    for r in risultati:
        display(Markdown(f"### 🎙️ Audio: `{r['nome']}`"))
        display(pd.DataFrame([r["dati"].get("Evento", {})]))


def mostra_anteprima_multi(dati):
    """Mostra l'anteprima del report multi-fonte."""
    display(Markdown("## 📋 MODALITÀ B — Anteprima report multi-audio"))
    display(Markdown("### Evento (sintesi incrociata)"))
    display(pd.DataFrame([dati.get("Evento", {})]))

# la funzione più utile per l'analisi: affianca su una stessa tabella le informazioni estratte da ciascun audio singolo (Modalità A) e il risultato consolidato (Modalità B), su colonne
def mostra_confronto(risultati_singoli, dati_multi):
    """Mostra una tabella di confronto affiancata delle sezioni chiave tra le due modalità."""
    display(Markdown("## 🔍 CONFRONTO — Dinamica Sintesi per modalità"))
    righe = []
    for r in risultati_singoli:
        righe.append({
            "Fonte": r["nome"],
            "Modalità": "A — Singolo",
            "Tipo Evento": r["dati"].get("Evento", {}).get("Tipo_Evento", "N/D"),
            "Gravità": r["dati"].get("Evento", {}).get("Gravita_Stimata", "N/D"),
            "Dinamica (Sintesi)": r["dati"].get("Evento", {}).get("Dinamica_Sintesi", "N/D"),
            "Causa Radice": r["dati"].get("Analisi_Cause", {}).get("Causa_Radice", "N/D"),
        })
    righe.append({
        "Fonte": f"{len(risultati_singoli)} audio combinati",
        "Modalità": "B — Multi-fonte",
        "Tipo Evento": dati_multi.get("Evento", {}).get("Tipo_Evento", "N/D"),
        "Gravità": dati_multi.get("Evento", {}).get("Gravita_Stimata", "N/D"),
        "Dinamica (Sintesi)": dati_multi.get("Evento", {}).get("Dinamica_Sintesi", "N/D"),
        "Causa Radice": dati_multi.get("Analisi_Cause", {}).get("Causa_Radice", "N/D"),
    })
    df_confronto = pd.DataFrame(righe)
    display(df_confronto)
    return df_confronto

print("✅ Funzioni anteprima pronte.")

✅ Funzioni anteprima pronte.


In [12]:
# =====================================================================
# CELLA 12: ▶️ ESEGUI — MODALITÀ A (analisi singola per ogni audio)
# =====================================================================

mancanti = [f for f in FILE_AUDIO if not os.path.exists(f)]
if mancanti:
    print(f"❌ File mancanti: {mancanti}")
else:
    # --- ANALISI ---
    risultati_singoli = analizza_singoli(FILE_AUDIO)

    # --- ANTEPRIMA ---
    mostra_anteprima_singoli(risultati_singoli)

    # --- SALVATAGGIO EXCEL ---
    file_singoli = salva_excel_tutti_singoli(risultati_singoli)

    print(f"\n🎉 Modalità A completata! {len(file_singoli)} file Excel in '{OUTPUT_DIR}/'")


🎙️ [1/3] Analisi singola: 'Elena RSPP.mp3'
   ⬆️  Upload: Elena RSPP.mp3
   ⏳ Attendo elaborazione ✅
   🧠 Estrazione dati...
   ✅ Completato.

🎙️ [2/3] Analisi singola: 'Marco Capolinea.mp3'
   ⬆️  Upload: Marco Capolinea.mp3
   ⏳ Attendo elaborazione ✅
   🧠 Estrazione dati...
   ✅ Completato.

🎙️ [3/3] Analisi singola: 'Martina Operatrice.mp3'
   ⬆️  Upload: Martina Operatrice.mp3
   ⏳ Attendo elaborazione ✅
   🧠 Estrazione dati...
   ✅ Completato.

✅ Modalità A completata: 3 audio analizzati singolarmente.


## 📋 MODALITÀ A — Anteprima per audio

### 🎙️ Audio: `Elena RSPP.mp3`

,id_evento,Data_Ora,Tipo_Evento,Gravita_Stimata,Prima_dell_evento,Trigger_Evento_Principale,Dinamica_Sintesi
0,INC-AAAA-NNN,GG/MM/AAAA HH:MM,Incidente,Bassa,Intervento manuale su una macchina in funzione,Introduzione della mano dell'operatore in una ...,Un operatore ha introdotto la mano in una zona...


### 🎙️ Audio: `Marco Capolinea.mp3`

,id_evento,Data_Ora,Tipo_Evento,Gravita_Stimata,Prima_dell_evento,Trigger_Evento_Principale,Dinamica_Sintesi
0,N/D,N/D 10:15,Incidente,Bassa,La stazione di termosaldatura sulla linea 3 pr...,Intervento manuale di un operatore per riallin...,"Intorno alle 10:15, un operatore è intervenuto..."


### 🎙️ Audio: `Martina Operatrice.mp3`

,id_evento,Data_Ora,Tipo_Evento,Gravita_Stimata,Prima_dell_evento,Trigger_Evento_Principale,Dinamica_Sintesi
0,N/D,N/D,Incidente,Bassa,La termosaldatrice si bloccava continuamente a...,La piastra della termosaldatrice è scesa inasp...,"Durante il turno di mattina, la termosaldatric..."



📊 Modalità A — salvataggio 3 file Excel individuali...
   💾 Salvato: 'output_report\report_singolo_Elena RSPP.xlsx'
   💾 Salvato: 'output_report\report_singolo_Marco Capolinea.xlsx'
   💾 Salvato: 'output_report\report_singolo_Martina Operatrice.xlsx'
✅ 3 file Excel salvati nella cartella 'output_report'.

🎉 Modalità A completata! 3 file Excel in 'output_report/'


In [13]:
# =====================================================================
# CELLA 13: ▶️ ESEGUI — MODALITÀ B (fusione dei risultati Modalità A)
# ⚠️  Esegui PRIMA la Cella 13 (Modalità A)
# =====================================================================

try:
    # Verifichiamo che i dati della Modalità A siano presenti
    risultati_singoli
except NameError:
    print("❌ Errore: 'risultati_singoli' non trovato. Esegui la Cella 13 prima!")
else:
    # 1. Eseguiamo la fusione intelligente tramite Gemini
    dati_multi = fondi_risultati_singoli(risultati_singoli)

    # 2. Anteprima a video del risultato unificato
    mostra_anteprima_multi(dati_multi)

    # 3. Salvataggio definitivo nel file Excel unificato
    file_multi = salva_excel_multi(dati_multi)

    print(f"\n🎉 Modalità B completata con successo!")
    print(f"📊 Report finale unificato creato: '{file_multi}'")


🧠 Avvio fusione intelligente di 3 fonti con Gemini...


## 📋 MODALITÀ B — Anteprima report multi-audio

### Evento (sintesi incrociata)

,id_evento,Data_Ora,Tipo_Evento,Gravita_Stimata,Prima_dell_evento,Trigger_Evento_Principale,Dinamica_Sintesi
0,INC-AAAA-NNN,GG/MM/AAAA 10:15,Incidente,Bassa,La stazione di termosaldatura sulla Linea 3 pr...,La piastra della termosaldatrice è scesa inasp...,"Durante il turno di mattina, intorno alle 10:1..."



📊 Modalità B — salvataggio report multi-audio: 'output_report\report_multiaudio.xlsx'...
✅ Report multi-audio salvato: 'output_report\report_multiaudio.xlsx'

🎉 Modalità B completata con successo!
📊 Report finale unificato creato: 'output_report\report_multiaudio.xlsx'
